# 03 — Modelado y Evaluación
## Network Anomaly Detection — NSL-KDD

**Metodología:** CRISP-DM — Fases 4 y 5: Modeling + Evaluation

### Modelos
| # | Modelo | Framework | Rol |
|---|--------|-----------|-----|
| 1 | Random Forest | PySpark MLlib | Clasificador principal |
| 2 | GBT One-vs-Rest | PySpark MLlib | Clasificador alternativo |
| 3 | Autoencoder | Keras | Detector de anomalías auxiliar |

### Reglas críticas
- El Autoencoder se entrena **SOLO** con tráfico Normal de train
- El threshold del Autoencoder se calibra en **validation**, nunca en test
- El test se evalúa **UNA SOLA VEZ** al final — sin re-tuning
- Accuracy **NO** es la métrica principal (desbalance de clases)

## 0. Setup

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')
os.environ['JAVA_HOME'] = '/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from train_models import RandomForestModel, GBTModel, AutoencoderModel, ModelEvaluator

SEED = 42
np.random.seed(SEED)

spark = SparkSession.builder \
    .appName('NetworkAnomalyDetection_Modeling') \
    .master('local[*]') \
    .config('spark.driver.memory', '6g') \
    .config('spark.ui.showConsoleProgress', 'false') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print(f'PySpark {spark.version} — {spark.sparkContext.defaultParallelism} cores')

# Cargar metadata del pipeline
with open('../configs/pipeline_meta.json') as f:
    meta = json.load(f)

label_mapping = meta['label_mapping']
weight_map    = {int(k): v for k, v in meta['weight_map'].items()}
print(f'Clases: {label_mapping}')

## 1. Cargar datos procesados

In [ ]:
import yaml
with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

base = '../data/processed'
train_df = spark.read.parquet(f'{base}/train_features.parquet')
val_df   = spark.read.parquet(f'{base}/val_features.parquet')
test_df  = spark.read.parquet(f'{base}/test_features.parquet')

print(f'Train: {train_df.count():,} | Val: {val_df.count():,} | Test: {test_df.count():,}')
train_df.printSchema()

## 2. Evaluador compartido

In [ ]:
evaluator = ModelEvaluator(label_mapping)
all_results = []  # acumulador para tabla comparativa final

def plot_confusion_matrix(cm, title):
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                linewidths=0.5, ax=ax)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_ylabel('Real')
    ax.set_xlabel('Predicho')
    plt.tight_layout()
    fname = title.lower().replace(' ', '_').replace('(','').replace(')','') + '.png'
    plt.savefig(f'../reports/{fname}', dpi=150, bbox_inches='tight')
    plt.show()

## 3. Modelo 1 — Random Forest

In [ ]:
rf_model = RandomForestModel(config, weight_map, seed=SEED)
rf_model.train(train_df, val_df)

In [ ]:
# Evaluar en validation
rf_preds_val = rf_model.predict(val_df)
rf_val_results = evaluator.evaluate(rf_preds_val, 'Val')
rf_val_results['model'] = 'Random Forest'
all_results.append(rf_val_results)

print('\nRandom Forest — Validation:')
for k, v in rf_val_results.items():
    if k not in ('dataset', 'model'):
        print(f'  {k:30s}: {v}')

cm_rf = evaluator.confusion_matrix(rf_preds_val)
plot_confusion_matrix(cm_rf, 'Confusion Matrix — Random Forest (Val)')

In [ ]:
# Feature importance
fi = rf_model.feature_importance(meta['feature_cols'])
top20 = fi.head(20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20['feature'], top20['importance'], color='#2E74B5')
ax.set_title('Feature Importance — Random Forest (Top 20)', fontweight='bold')
ax.set_xlabel('Importancia')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/06_rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

rf_model.save('../models/random_forest')

## 4. Modelo 2 — Gradient Boosted Trees

In [ ]:
gbt_model = GBTModel(config, seed=SEED)
gbt_model.train(train_df, val_df)

In [ ]:
gbt_preds_val = gbt_model.predict(val_df)
gbt_val_results = evaluator.evaluate(gbt_preds_val, 'Val')
gbt_val_results['model'] = 'GBT'
all_results.append(gbt_val_results)

print('\nGBT — Validation:')
for k, v in gbt_val_results.items():
    if k not in ('dataset', 'model'):
        print(f'  {k:30s}: {v}')

cm_gbt = evaluator.confusion_matrix(gbt_preds_val)
plot_confusion_matrix(cm_gbt, 'Confusion Matrix — GBT (Val)')

gbt_model.save('../models/gbt')

## 5. Modelo 3 — Autoencoder (solo tráfico Normal)

In [ ]:
ae_model = AutoencoderModel(config, seed=SEED)
ae_model.train(train_df, val_df)

In [ ]:
# Evaluar Autoencoder en validation
ae_scores_val = ae_model.get_anomaly_scores(val_df)

# Métricas binarias (Normal vs Ataque)
tp = ((ae_scores_val['predicted_anomaly']) & (ae_scores_val['attack_category'] != 'Normal')).sum()
tn = ((~ae_scores_val['predicted_anomaly']) & (ae_scores_val['attack_category'] == 'Normal')).sum()
fp = ((ae_scores_val['predicted_anomaly']) & (ae_scores_val['attack_category'] == 'Normal')).sum()
fn = ((~ae_scores_val['predicted_anomaly']) & (ae_scores_val['attack_category'] != 'Normal')).sum()

recall_ae  = tp / (tp + fn + 1e-9)
prec_ae    = tp / (tp + fp + 1e-9)
f1_ae      = 2 * recall_ae * prec_ae / (recall_ae + prec_ae + 1e-9)

print(f'Autoencoder (Binario: Normal vs Ataque) — Validation:')
print(f'  Recall:    {recall_ae:.4f}')
print(f'  Precision: {prec_ae:.4f}')
print(f'  F1:        {f1_ae:.4f}')
print(f'  Threshold: {ae_model.threshold:.6f}')

# Distribución del anomaly score por clase
fig, ax = plt.subplots(figsize=(10, 5))
for cls in ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']:
    subset = ae_scores_val[ae_scores_val['attack_category'] == cls]['anomaly_score']
    if len(subset) > 0:
        ax.hist(subset, bins=50, alpha=0.5, label=cls, density=True)
ax.axvline(ae_model.threshold / (ae_model.max_mse + 1e-9),
           color='black', linestyle='--', linewidth=2, label='Threshold')
ax.set_title('Distribución del Anomaly Score por clase', fontweight='bold')
ax.set_xlabel('Anomaly Score (normalizado)')
ax.set_ylabel('Densidad')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/07_autoencoder_scores.png', dpi=150, bbox_inches='tight')
plt.show()

ae_val_results = {
    'model': 'Autoencoder', 'dataset': 'Val',
    'recall_binario': round(recall_ae, 4),
    'precision_binario': round(prec_ae, 4),
    'f1_binario': round(f1_ae, 4),
    'threshold': round(ae_model.threshold, 6)
}
ae_model.save('../models/autoencoder')

## 6. Tabla comparativa — Validation

In [ ]:
# Comparar RF vs GBT en validation
comp_cols = ['model', 'f1_macro', 'accuracy', 'weighted_recall',
             'recall_Normal', 'recall_DoS', 'recall_Probe', 'recall_R2L', 'recall_U2R']

comp_df = pd.DataFrame(all_results)[comp_cols]
print('\nComparativa en Validation (ordenada por F1-macro):')
print(comp_df.sort_values('f1_macro', ascending=False).to_string(index=False))

# Selección del mejor modelo para el agente RL
best_idx = comp_df['f1_macro'].idxmax()
best_model_name = comp_df.loc[best_idx, 'model']
print(f'\n✓ Mejor modelo seleccionado para el agente RL: {best_model_name}')
print('  (evaluación en test pendiente — una sola vez al final)')

# Guardar resultados de validation
comp_df.to_csv('../reports/model_comparison_val.csv', index=False)
print('✓ Guardado: reports/model_comparison_val.csv')

## 7. Evaluación FINAL en Test

> ⚠️ **Esta celda se ejecuta UNA SOLA VEZ.** No se permite re-tuning después de ver los resultados del test.

In [ ]:
print('EVALUACIÓN FINAL EN TEST — RF y GBT')
print('='*50)
print('⚠ Esta evaluación es definitiva. No hay re-tuning posterior.')
print()

final_results = []

# Random Forest en test
rf_preds_test = rf_model.predict(test_df)
rf_test = evaluator.evaluate(rf_preds_test, 'Test')
rf_test['model'] = 'Random Forest'
final_results.append(rf_test)

# GBT en test
gbt_preds_test = gbt_model.predict(test_df)
gbt_test = evaluator.evaluate(gbt_preds_test, 'Test')
gbt_test['model'] = 'GBT'
final_results.append(gbt_test)

final_df = pd.DataFrame(final_results)[comp_cols]
print('Resultados FINALES en Test:')
print(final_df.to_string(index=False))

# Overfitting check: comparar val vs test
print('\nVerificación de overfitting (Val → Test):')
for res_val, res_test in zip(all_results, final_results):
    drop = res_val['f1_macro'] - res_test['f1_macro']
    flag = '⚠ POSIBLE OVERFITTING' if drop > 0.05 else '✓ OK'
    print(f"  {res_val['model']:15s}: Val F1={res_val['f1_macro']:.4f} → Test F1={res_test['f1_macro']:.4f}  Δ={drop:+.4f}  {flag}")

# Matriz de confusión del mejor modelo en test
best_preds_test = rf_preds_test if best_model_name == 'Random Forest' else gbt_preds_test
cm_test = evaluator.confusion_matrix(best_preds_test)
plot_confusion_matrix(cm_test, f'Confusion Matrix — {best_model_name} (Test FINAL)')

# Guardar resultados finales
final_df.to_csv('../reports/model_comparison_test_FINAL.csv', index=False)
print('\n✓ Guardado: reports/model_comparison_test_FINAL.csv')
print('→ Siguiente paso: 04_rl_agent.ipynb')

In [ ]:
spark.stop()
print('✓ SparkSession cerrada')